# VPU 单元测试

测试 FPGA 上部署的 VPU 各功能单元，与 golden model 对比。

## 硬件配置
- XDMA 设备：/dev/xdma0
- VPU AXI Regs 基地址：0x1004_0000
- GB BRAM 基地址：0x1000_0000
- WB BRAM 基地址：0x1002_0000

In [1]:
import numpy as np
import sys
from pathlib import Path

# 添加VPU模块路径
sys.path.insert(0, str(Path.cwd().parent))

# 导入 xdma_helpers（使用 xdma_rw.exe）
from xdma_helpers import xdma_write, xdma_read, write_reg, read_reg, XDMA_EXE

# VPU 地址映射
VPU_REGS_BASE = 0x10040000
GLOBAL_BRAM_BASE = 0x10000000  # Staging global_bram
GB_BRAM_BASE = 0x10010000      # VPU Global Buffer
WB_BRAM_BASE = 0x10020000      # VPU Weight Buffer
CDMA_BASE = 0x10030000

# VPU 寄存器偏移
VPU_CTRL = 0x00
VPU_STATUS = 0x04
VPU_UNIT_CHOOSE = 0x08
VPU_SRC_ADDR = 0x0C
VPU_SRC2_ADDR = 0x10
VPU_SRC_C = 0x14
VPU_SRC_H = 0x18
VPU_SRC_W = 0x1C
VPU_BIAS_ADDR = 0x20
VPU_SCALE_ADDR = 0x24
VPU_DST_ADDR = 0x28
VPU_ADDR_BREAK = 0x2C
VPU_ADDR_S = 0x30
VPU_ADDR_T = 0x34

# VPU 单元编号
UNIT_DQA = 1  # Dequantization
UNIT_NN = 2   # NN_LUT (注释掉)
UNIT_QA = 3   # Quantization
UNIT_MP = 4   # Max Pooling
UNIT_US = 5   # Upsampling
UNIT_AD = 6   # Addition

print("✓ VPU 测试环境初始化完成")
print(f"✓ 使用工具: {XDMA_EXE}")
print(f"✓ GB BRAM: 0x{GB_BRAM_BASE:08x}")
print(f"✓ WB BRAM: 0x{WB_BRAM_BASE:08x}")
print(f"✓ VPU Regs: 0x{VPU_REGS_BASE:08x}")

VPU 测试环境初始化完成
GB BRAM: 0x10010000
WB BRAM: 0x10020000
VPU Regs: 0x10040000


## 1. XDMA 基础读写测试

In [2]:
import time

# 测试寄存器读写（使用 xdma_rw.exe）
print("测试 VPU 状态寄存器读取...")
status = read_reg(VPU_REGS_BASE, VPU_STATUS)
print(f"✓ VPU Status: 0x{status:08x} (ready={status & 1})")

测试 VPU 状态寄存器读取...


FileNotFoundError: [Errno 2] No such file or directory: '/dev/xdma0_c2h_0'

## 2. Global Buffer / Weight Buffer 测试

In [ ]:
def test_bram_rw():
    """测试 GB/WB BRAM 读写（使用 xdma_rw.exe）"""
    print("\n=== BRAM 读写测试 ===")
    
    # 测试 Global Buffer
    test_data = np.random.randint(0, 256, 64, dtype=np.uint8)
    xdma_write(GB_BRAM_BASE, test_data.tobytes())
    read_back = np.frombuffer(xdma_read(GB_BRAM_BASE, 64), dtype=np.uint8)
    
    if np.array_equal(test_data, read_back):
        print("✅ Global Buffer 读写正确")
    else:
        print("❌ Global Buffer 读写错误")
        print(f"写入: {test_data[:8]}")
        print(f"读回: {read_back[:8]}")

test_bram_rw()

## 3. MP Unit (Max Pooling) 测试

In [ ]:
def test_mp_unit():
    """测试 Max Pooling 单元"""
    print("\n=== MP Unit 测试 ===")
    
    # Golden model
    def golden_max_pool_2x2(data, C, H, W):
        data_3d = data.reshape(C, H, W)
        out_h, out_w = H // 2, W // 2
        output = np.zeros((C, out_h, out_w), dtype=data.dtype)
        for c in range(C):
            for h in range(out_h):
                for w in range(out_w):
                    pool = data_3d[c, h*2:h*2+2, w*2:w*2+2]
                    output[c, h, w] = pool.max()
        return output.flatten()
    
    # 准备测试数据
    C, H, W = 4, 4, 4
    input_data = np.random.randint(0, 100, C*H*W, dtype=np.uint8)
    
    # 写入 GB BRAM
    xdma_write(GB_BRAM_BASE, input_data.tobytes())
    
    # 配置 VPU - 使用正确的 UNIT_MP = 4
    write_reg(VPU_REGS_BASE, VPU_UNIT_CHOOSE, 4)  # UNIT_MP
    write_reg(VPU_REGS_BASE, VPU_SRC_ADDR, 0)
    write_reg(VPU_REGS_BASE, VPU_DST_ADDR, 512)
    write_reg(VPU_REGS_BASE, VPU_SRC_C, C)
    write_reg(VPU_REGS_BASE, VPU_SRC_H, H)
    write_reg(VPU_REGS_BASE, VPU_SRC_W, W)
    
    # 启动
    write_reg(VPU_REGS_BASE, VPU_CTRL, 1)
    
    # 等待完成
    for _ in range(100):
        status = read_reg(VPU_REGS_BASE, VPU_STATUS)
        if status & 1:
            break
        time.sleep(0.01)
    
    write_reg(VPU_REGS_BASE, VPU_CTRL, 0)
    
    # 读取结果
    out_size = C * (H//2) * (W//2)
    hw_output = np.frombuffer(xdma_read(GB_BRAM_BASE + 512, out_size), dtype=np.uint8)
    
    # 对比 golden
    golden_output = golden_max_pool_2x2(input_data, C, H, W)
    
    if np.array_equal(hw_output, golden_output):
        print("✅ MP Unit 测试通过")
    else:
        print("❌ MP Unit 测试失败")
        print(f"Golden: {golden_output[:8]}")
        print(f"HW:     {hw_output[:8]}")

test_mp_unit()

## 4. QA Unit (Quantization) 测试

In [ ]:
def test_qa_unit():
    """测试 FP32 → INT8 量化单元"""
    print("\n=== QA Unit 测试 ===")
    
    # Golden model
    def golden_quantize_fp32_to_int8(fp32_data, scale):
        return np.clip(np.round(fp32_data * scale), -128, 127).astype(np.int8)
    
    # 准备测试数据（FP32）
    N = 32
    fp32_data = np.random.randn(N).astype(np.float32) * 10.0
    scale = 12.8  # 缩放因子
    
    # 写入 GB BRAM (FP32 格式)
    xdma_write(GB_BRAM_BASE, fp32_data.tobytes())
    
    # 写入 scale 到 WB BRAM
    scale_array = np.array([scale], dtype=np.float32)
    xdma_write(WB_BRAM_BASE, scale_array.tobytes())
    
    # 配置 VPU
    write_reg(VPU_REGS_BASE, VPU_UNIT_CHOOSE, 4)  # UNIT_QA
    write_reg(VPU_REGS_BASE, VPU_SRC_ADDR, 0)
    write_reg(VPU_REGS_BASE, 0x24, 0)  # scale_addr
    write_reg(VPU_REGS_BASE, VPU_DST_ADDR, 1024)
    write_reg(VPU_REGS_BASE, 0x14, N)  # src_c
    
    # 启动并等待
    write_reg(VPU_REGS_BASE, VPU_CTRL, 1)
    time.sleep(0.1)
    write_reg(VPU_REGS_BASE, VPU_CTRL, 0)
    
    # 读取结果
    hw_output = np.frombuffer(xdma_read(GB_BRAM_BASE + 1024, N), dtype=np.int8)
    
    # 对比 golden
    golden_output = golden_quantize_fp32_to_int8(fp32_data, scale)
    
    diff = np.abs(hw_output.astype(np.int16) - golden_output.astype(np.int16))
    max_err = diff.max()
    
    if max_err <= 1:  # 允许 ±1 误差
        print(f"✅ QA Unit 测试通过 (max_err={max_err})")
    else:
        print(f"❌ QA Unit 测试失败 (max_err={max_err})")

test_qa_unit()

## 5. DQA Unit (Dequantization) 测试

In [ ]:
def test_dqa_unit():
    """测试 INT32 → FP32 反量化单元"""
    print("\n=== DQA Unit 测试 ===")
    
    # Golden model: out_fp32 = int32_data * scale + bias
    def golden_dequantize_int32_to_fp32(int32_data, scale, bias):
        return int32_data.astype(np.float32) * scale + bias
    
    # 准备测试数据（INT32）
    N = 32
    int32_data = np.random.randint(-1000, 1000, N, dtype=np.int32)
    scale = 0.1
    bias = 5.0
    
    # 写入 INT32 数据到 GB BRAM
    xdma_write(GB_BRAM_BASE, int32_data.tobytes())
    
    # 写入 scale 和 bias 到 WB BRAM
    scale_array = np.full(N, scale, dtype=np.float32)
    bias_array = np.full(N, bias, dtype=np.float32)
    xdma_write(WB_BRAM_BASE, scale_array.tobytes())
    xdma_write(WB_BRAM_BASE + 256, bias_array.tobytes())
    
    # 配置 VPU
    write_reg(VPU_REGS_BASE, VPU_UNIT_CHOOSE, 5)  # UNIT_DQA
    write_reg(VPU_REGS_BASE, VPU_SRC_ADDR, 0)
    write_reg(VPU_REGS_BASE, 0x24, 0)  # scale_addr
    write_reg(VPU_REGS_BASE, 0x20, 256)  # bias_addr
    write_reg(VPU_REGS_BASE, VPU_DST_ADDR, 2048)
    write_reg(VPU_REGS_BASE, 0x14, N)  # src_c
    
    # 启动并等待
    write_reg(VPU_REGS_BASE, VPU_CTRL, 1)
    time.sleep(0.1)
    write_reg(VPU_REGS_BASE, VPU_CTRL, 0)
    
    # 读取结果（FP32）
    hw_output = np.frombuffer(xdma_read(GB_BRAM_BASE + 2048, N*4), dtype=np.float32)
    
    # 对比 golden
    golden_output = golden_dequantize_int32_to_fp32(int32_data, scale, bias)
    
    diff = np.abs(hw_output - golden_output)
    max_err = diff.max()
    
    if max_err < 1e-5:  # FP32 精度
        print(f"✅ DQA Unit 测试通过 (max_err={max_err:.2e})")
    else:
        print(f"❌ DQA Unit 测试失败 (max_err={max_err:.2e})")
        print(f"Sample - Golden: {golden_output[:4]}")
        print(f"Sample - HW:     {hw_output[:4]}")

test_dqa_unit()

## 6. US Unit (Upsampling) 测试

In [ ]:
def test_us_unit():
    """测试上采样单元（最近邻插值 2x）"""
    print("\n=== US Unit 测试 ===")
    
    # Golden model: 最近邻上采样 2x
    def golden_upsample_2x(data, C, H, W):
        data_3d = data.reshape(C, H, W)
        output = np.zeros((C, H*2, W*2), dtype=data.dtype)
        for c in range(C):
            for h in range(H):
                for w in range(W):
                    output[c, h*2:h*2+2, w*2:w*2+2] = data_3d[c, h, w]
        return output.flatten()
    
    # 准备测试数据
    C, H, W = 4, 2, 2
    input_data = np.random.randint(0, 100, C*H*W, dtype=np.uint8)
    
    # 写入 GB BRAM
    xdma_write(GB_BRAM_BASE, input_data.tobytes())
    
    # 配置 VPU
    write_reg(VPU_REGS_BASE, VPU_UNIT_CHOOSE, 6)  # UNIT_US
    write_reg(VPU_REGS_BASE, VPU_SRC_ADDR, 0)
    write_reg(VPU_REGS_BASE, VPU_DST_ADDR, 512)
    write_reg(VPU_REGS_BASE, 0x14, C)  # src_c
    write_reg(VPU_REGS_BASE, 0x18, H)  # src_h
    write_reg(VPU_REGS_BASE, 0x1C, W)  # src_w
    
    # 启动并等待
    write_reg(VPU_REGS_BASE, VPU_CTRL, 1)
    for _ in range(100):
        status = read_reg(VPU_REGS_BASE, VPU_STATUS)
        if status & 1:
            break
        time.sleep(0.01)
    write_reg(VPU_REGS_BASE, VPU_CTRL, 0)
    
    # 读取结果
    out_size = C * (H*2) * (W*2)
    hw_output = np.frombuffer(xdma_read(GB_BRAM_BASE + 512, out_size), dtype=np.uint8)
    
    # 对比 golden
    golden_output = golden_upsample_2x(input_data, C, H, W)
    
    if np.array_equal(hw_output, golden_output):
        print("✅ US Unit 测试通过")
    else:
        print("❌ US Unit 测试失败")
        print(f"Golden shape: {golden_output.shape}")
        print(f"HW shape:     {hw_output.shape}")
        print(f"Golden: {golden_output[:16]}")
        print(f"HW:     {hw_output[:16]}")

test_us_unit()

## 7. AD Unit (Element-wise Add) 测试

In [ ]:
def test_ad_unit():
    """测试 FP32 element-wise 加法单元"""
    print("\n=== AD Unit 测试 ===")
    
    # Golden model: a + b (element-wise)
    def golden_fp32_add(a, b):
        return a + b
    
    # 准备测试数据（FP32）
    N = 64
    a_data = np.random.randn(N).astype(np.float32)
    b_data = np.random.randn(N).astype(np.float32)
    
    # 写入 GB BRAM（a 在偏移 0，b 在偏移 256）
    xdma_write(GB_BRAM_BASE, a_data.tobytes())
    xdma_write(GB_BRAM_BASE + 256, b_data.tobytes())
    
    # 配置 VPU
    write_reg(VPU_REGS_BASE, VPU_UNIT_CHOOSE, 7)  # UNIT_AD
    write_reg(VPU_REGS_BASE, VPU_SRC_ADDR, 0)  # src (a)
    write_reg(VPU_REGS_BASE, 0x10, 256)  # src2_addr (b)
    write_reg(VPU_REGS_BASE, VPU_DST_ADDR, 2048)
    write_reg(VPU_REGS_BASE, 0x14, N)  # src_c
    
    # 启动并等待
    write_reg(VPU_REGS_BASE, VPU_CTRL, 1)
    time.sleep(0.1)
    write_reg(VPU_REGS_BASE, VPU_CTRL, 0)
    
    # 读取结果
    hw_output = np.frombuffer(xdma_read(GB_BRAM_BASE + 2048, N*4), dtype=np.float32)
    
    # 对比 golden
    golden_output = golden_fp32_add(a_data, b_data)
    
    diff = np.abs(hw_output - golden_output)
    max_err = diff.max()
    
    if max_err < 1e-5:
        print(f"✅ AD Unit 测试通过 (max_err={max_err:.2e})")
    else:
        print(f"❌ AD Unit 测试失败 (max_err={max_err:.2e})")

test_ad_unit()

## 5. Im2Col + MAC 测试

In [ ]:
# 导入 im2col 测试模块
import sys
sys.path.insert(0, '/home/triton/task/YOLO_on_FPGA/fpga/EdgeYOLO-FPGA/tests/vpu')

from im2col import host_im2col, golden_conv1x1_int32, conv_weight_to_gemm_tile

def test_im2col_mac():
    """测试 im2col + MAC 阵列"""
    print("\n=== Im2Col + MAC 测试 ===")
    
    np.random.seed(42)
    C_in, C_out = 16, 8
    H, W = 4, 4
    
    # 准备权重和激活
    weight = np.random.randint(-8, 8, size=(C_out, C_in, 1, 1), dtype=np.int8)
    activation = np.random.randint(-64, 64, size=(C_in, H, W), dtype=np.int8)
    
    # Host im2col
    im2col_matrix = host_im2col(activation, 1, 1, 0, 1)
    print(f"Im2col 矩阵形状: {im2col_matrix.shape}")
    
    # Golden model
    golden_output = golden_conv1x1_int32(weight, activation)
    print(f"Golden 输出形状: {golden_output.shape}")
    
    # TODO: 在 FPGA 上运行 MAC 阵列
    # 这里需要配置 DQA/MAC 单元来执行矩阵乘法
    
    print("✅ Im2Col 测试完成 (仅 golden model)")

test_im2col_mac()

## 总结

本 notebook 提供了 VPU 各单元的完整测试框架。

### 已实现测试单元
- ✅ **XDMA 读写** - PCIe 通信基础测试
- ✅ **BRAM 读写** - Global Buffer / Weight Buffer 测试
- ✅ **MP Unit** - Max Pooling 2x2 池化
- ✅ **QA Unit** - FP32 → INT8 量化
- ✅ **DQA Unit** - INT32 → FP32 反量化（scale + bias）
- ✅ **US Unit** - 2x 最近邻上采样
- ✅ **AD Unit** - FP32 element-wise 加法
- ✅ **Im2Col** - Host 端 im2col 展开

### 单元编号（unit_choose）
- 0: NN_LUT (暂未测试)
- 1: SU (暂未测试)
- 2: MP - Max Pooling
- 3: (reserved)
- 4: QA - Quantization
- 5: DQA - Dequantization
- 6: US - Upsampling
- 7: AD - Addition

### 使用说明
1. 确保 FPGA 已烧录 VPU 比特流
2. 加载 XDMA 驱动：`sudo modprobe xdma`
3. 逐个运行测试单元
4. 每个测试会输出 ✅ 通过或 ❌ 失败

### 待完善
- MAC 阵列完整测试（矩阵乘法）
- 完整卷积流程测试（im2col + MAC + pooling）
- 多单元组合测试
- 性能测试（吞吐量、延迟）